# MUC01 — Retail Sales Category Intelligence EDA

## Team Final Notebook

This notebook combines the three team members' analysis into one connected project, preserving the substantive analysis while removing exact duplicate sections.

### Analysis flow
1. CBIM Problem Canvas and dataset understanding
2. Data inspection and preparation
3. Category revenue trend and consistency analysis
4. Top and bottom products by category
5. Discount effectiveness analysis
6. Day-of-week and monthly sales patterns
7. Electronics monthly and weekday analysis
8. Additional total-revenue-by-day analysis
9. Business interpretation and final observations

**Common dataset:** `MUC01_Retail_Sales_Dataset.csv`


# CBIM Problem Canvas

### Situation (S)
A regional retail chain operates 12 stores across Andhra Pradesh and sells products across five categories: Electronics, Apparel, Grocery, Home & Kitchen, and Personal Care. The Category Manager is preparing for a quarterly supplier review and needs a clear understanding of category and product performance.

### Complication (C)
The Category Manager needs to decide which categories deserve continued or increased shelf space and promotional investment, but she needs evidence from the six months of transaction data to identify growing, declining, and consistently performing categories.

### Question (Q)
Which product categories are growing, declining, or consistently performing over the six-month period, and which categories should the Category Manager prioritise or deprioritise during the supplier review?

### SSOT
MUC01_Retail_Sales_Dataset.csv, containing six months of retail transaction data from January to June 2026.

### Success Definition
The analysis will clearly identify the growing, declining, and most consistent categories using revenue trends and business interpretation, enabling the Category Manager to make informed supplier and investment decisions.

### Primary Stakeholder
The Category Manager, who needs a data-backed view of category performance to make decisions about shelf space, promotional investment, and supplier priorities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("MUC01_Retail_Sales_Dataset.csv")

display(df.head())

In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
display(df.head())

In [ ]:
print(df.dtypes)

In [ ]:
print(df.isnull().sum())

In [ ]:
display(df.describe())

In [ ]:
print("Number of unique products:", df['product_name'].nunique())

In [ ]:
print("Number of unique categories:", df['category'].nunique())

In [ ]:
print("Categories:", df['category'].unique())

Step 5.1 — Data Inspection

The dataset contains 107,836 retail transactions across 10 columns, covering January to June 2026. There are 25 unique products across 5 product categories, with no missing values in any column.

The average transaction contains approximately 2.6 units, suggesting that customers generally purchase multiple units per transaction and that bundle or cross-selling opportunities may exist. The average transaction revenue is approximately ₹4,264, while the maximum transaction revenue is approximately ₹3.55 lakh, indicating substantial variation in transaction value, potentially due to high-priced products.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.dtypes

In [ ]:
df['month'] = df['date'].dt.to_period('M')

In [ ]:
df[['date', 'month']].head()

In [ ]:
monthly_category_revenue = (
    df.groupby(['month', 'category'])['revenue']
      .sum()
      .reset_index()
)

In [ ]:
monthly_category_revenue.head(10)

In [ ]:
monthly_category_revenue = (
    df.groupby(['month', 'category'])['revenue']
      .sum()
      .reset_index()
)

In [ ]:
monthly_category_revenue

In [ ]:
monthly_category_revenue['month'] = monthly_category_revenue['month'].astype(str)

In [ ]:
monthly_category_revenue.dtypes

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly_category_revenue,
    x='month',
    y='revenue',
    hue='category',
    marker='o'
)

plt.title('Monthly Revenue Trend by Product Category')
plt.xlabel('Month')
plt.ylabel('Total Revenue (₹)')
plt.xticks(rotation=45)
plt.legend(title='Product Category')
plt.tight_layout()

plt.show()

In [ ]:
category_change = (
    monthly_category_revenue
    .pivot(index='month', columns='category', values='revenue')
)

category_change

In [ ]:
percentage_change = (
    (category_change.iloc[-1] - category_change.iloc[0])
    / category_change.iloc[0]
) * 100

percentage_change.sort_values()

In [ ]:
consistency = (
    category_change.std() / category_change.mean()
) * 100

consistency.sort_values()

In [ ]:
category_change = (
    monthly_category_revenue
    .pivot(index='month', columns='category', values='revenue')
)

category_change

In [ ]:
percentage_change = (
    (category_change.iloc[-1] - category_change.iloc[0])
    / category_change.iloc[0]
) * 100

percentage_change.sort_values()

In [ ]:
consistency = (
    category_change.std() / category_change.mean()
) * 100

consistency.sort_values()

Step 5.2 — Category Revenue Trend

Business Interpretation:
The monthly revenue trends show clear differences across the five categories. Electronics has experienced the steepest decline, falling from ₹4.57 crore in January to ₹2.97 crore in June, representing a 35.02% decrease. This declining trend is a warning signal for the Category Manager, and the Electronics supplier should be challenged to explain the decline before additional shelf space or promotional investment is committed.

Grocery is the most consistent category, with a coefficient of variation of only 3.87% across the six months. Its monthly revenue remained relatively stable, ranging from approximately ₹65 lakh to ₹73 lakh, making it a predictable contributor to overall revenue and a strong category for maintaining stable shelf presence.

Apparel also shows a strong positive trend, increasing by 34.10% from January to June, suggesting growing customer demand and a potential case for increased attention or investment.

Member 1 findings:
📉 Electronics declined by 35.02% from January to June and is the steepest declining category.
📈 Apparel increased by 34.10%, showing the strongest growth.
📊 Grocery is the most consistent category with a 3.87% coefficient of variation.

# Step 5.3 — Top and Bottom Products by Category

In this step, we identify the **top 2 and bottom 2 products by total revenue within each product category** over the six-month period.

The purpose is to understand which products are driving category performance and which products may need attention during the supplier review.

In [ ]:
product_revenue = (
    df.groupby(['category', 'product_name'])['revenue']
      .sum()
      .reset_index()
)

product_revenue = product_revenue.rename(
    columns={'revenue': 'total_revenue'}
)

display(product_revenue.head(10))

In [ ]:
product_revenue_desc = product_revenue.sort_values(
    ['category', 'total_revenue'],
    ascending=[True, False]
)

top_2_products = (
    product_revenue_desc
    .groupby('category', group_keys=False)
    .head(2)
)

display(top_2_products)

In [ ]:
product_revenue_asc = product_revenue.sort_values(
    ['category', 'total_revenue'],
    ascending=[True, True]
)

bottom_2_products = (
    product_revenue_asc
    .groupby('category', group_keys=False)
    .head(2)
)

display(bottom_2_products)

In [ ]:
top_table = top_2_products.copy()
top_table['performance'] = 'Top 2'

bottom_table = bottom_2_products.copy()
bottom_table['performance'] = 'Bottom 2'

product_summary = pd.concat(
    [top_table, bottom_table],
    ignore_index=True
)

product_summary = product_summary.sort_values(
    ['category', 'performance', 'total_revenue'],
    ascending=[True, True, False]
)

display(product_summary)

### Step 5.3 — Business Interpretation

The product-level analysis shows that **Electronics is the largest revenue-generating category**, but it also needs the closest review because the category has already shown the steepest decline in Step 5.2.

Within Electronics, **Headphones (₹47.70 million)** and **Laptop (₹46.55 million)** are the strongest products. In contrast, **Smart TV (₹41.88 million)** and **Tablet (₹44.74 million)** are the lowest two products in the category.

For the supplier review, **Electronics should be the category to challenge**. It contributes the highest total revenue, but its declining category trend means the supplier needs to explain how it will protect and improve performance. The weaker products, particularly Smart TV, should receive additional attention before further promotional or shelf-space investment is approved.

# Step 5.4 — Discount Effectiveness Analysis

The assignment asks us to compare transactions at **0%, 5%, 10%, 15%, 20%, and 25%+ discount levels**.

We will calculate:
- Average revenue per transaction
- Average units sold per transaction
- Number of transactions in each discount bracket

Then we will use a bar chart and the numbers to decide whether higher discounts are producing a meaningful return.

In [ ]:
def create_discount_bracket(discount):
    if discount == 0:
        return '0%'
    elif discount == 5:
        return '5%'
    elif discount == 10:
        return '10%'
    elif discount == 15:
        return '15%'
    elif discount == 20:
        return '20%'
    else:
        return '25%+'

df['discount_bracket'] = df['discount_pct'].apply(create_discount_bracket)

discount_counts = (
    df['discount_bracket']
    .value_counts()
    .reindex(['0%', '5%', '10%', '15%', '20%', '25%+'], fill_value=0)
)

display(discount_counts.to_frame('transaction_count'))

In [ ]:
discount_analysis = (
    df.groupby('discount_bracket')
      .agg(
          average_revenue=('revenue', 'mean'),
          average_units_sold=('units_sold', 'mean'),
          transaction_count=('transaction_id', 'count')
      )
      .reindex(['0%', '5%', '10%', '15%', '20%', '25%+'])
)

display(discount_analysis)

In [ ]:
discount_plot = (
    discount_analysis
    .dropna(subset=['average_revenue'])
    .reset_index()
)

plt.figure(figsize=(9, 5))

plt.bar(
    discount_plot['discount_bracket'],
    discount_plot['average_revenue'],
       
   color="#2D1139"  
)


plt.title('Average Revenue per Transaction by Discount Bracket')
plt.xlabel('Discount Bracket')
plt.ylabel('Average Revenue per Transaction (₹)')

plt.tight_layout()
plt.show()

### Step 5.4 — Business Interpretation

The discount analysis gives a clear signal: **higher discounts are not generating higher revenue per transaction**.

Average revenue per transaction falls from **₹4,685.65 at 0% discount** to **₹4,303.73 at 5%**, **₹4,005.55 at 10%**, and **₹3,631.73 at 20%**. Average units sold remain almost unchanged, ranging from about **2.57 to 2.60 units** across the available discount levels.

Therefore, the current discount pattern does not show a proportionate increase in units sold or revenue per transaction. The Category Manager should challenge suppliers to justify deeper discounts with evidence of incremental volume, rather than assuming that a larger discount automatically improves performance.

**Note:** There are no transactions in the dataset with a discount of 25% or more, so the 25%+ bracket has no calculated average.

# Step 5.5 — Day-of-Week and Monthly Sales Patterns

This step looks at when customers naturally generate more or less revenue.

First, we will calculate **average transaction revenue by day of week** for the whole retail chain. Then we will calculate **total revenue by month**.

Because Step 5.2 identified **Electronics as the most interesting category due to its steep decline**, we will repeat the day-of-week and monthly analysis for Electronics and compare its pattern with the overall chain.

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df['day_of_week'] = df['date'].dt.day_name()

day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

daily_revenue = (
    df.groupby('day_of_week')['revenue']
      .mean()
      .reindex(day_order)
)

display(daily_revenue.to_frame('average_revenue_per_transaction'))

In [ ]:
plt.figure(figsize=(10, 6)) 
 
plt.bar( 
    daily_revenue.index, 
    daily_revenue.values,
   
    color="#2D1139"
) 
 
plt.title('Average Revenue per Transaction by Day of Week') 
plt.xlabel('Day of Week') 
plt.ylabel('Average Revenue per Transaction (₹)') 
plt.xticks(rotation=30) 
plt.tight_layout() 
 
plt.show()

In [ ]:
best_day = daily_revenue.idxmax()
best_day_value = daily_revenue.max()

worst_day = daily_revenue.idxmin()
worst_day_value = daily_revenue.min()

print(f"Best day: {best_day} — ₹{best_day_value:,.2f} average revenue per transaction")
print(f"Worst day: {worst_day} — ₹{worst_day_value:,.2f} average revenue per transaction")

In [ ]:
df['month'] = df['date'].dt.to_period('M')

monthly_revenue = (
    df.groupby('month')['revenue']
      .sum()
)

display(monthly_revenue.to_frame('total_revenue'))

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    monthly_revenue.index.astype(str),
    monthly_revenue.values,
    color="#2D1139"
)

plt.title('Total Revenue by Month')
plt.xlabel('Month')
plt.ylabel('Total Revenue (₹)')
plt.tight_layout()

plt.show()

In [ ]:
best_month = monthly_revenue.idxmax()
best_month_value = monthly_revenue.max()

worst_month = monthly_revenue.idxmin()
worst_month_value = monthly_revenue.min()

print(f"Best month: {best_month} — ₹{best_month_value:,.2f}")
print(f"Worst month: {worst_month} — ₹{worst_month_value:,.2f}")

In [ ]:
interesting_category = 'Electronics'
category_df = df[df['category'] == interesting_category].copy()

category_daily_revenue = (
    category_df.groupby('day_of_week')['revenue']
               .mean()
               .reindex(day_order)
)

display(
    category_daily_revenue.to_frame(
        'average_revenue_per_transaction'
    )
)

In [ ]:
category_monthly_revenue = (
    category_df.groupby('month')['revenue']
               .sum()
)

display(category_monthly_revenue.to_frame('total_revenue'))

In [ ]:
category_best_day = category_daily_revenue.idxmax()
category_best_day_value = category_daily_revenue.max()

category_worst_day = category_daily_revenue.idxmin()
category_worst_day_value = category_daily_revenue.min()

category_best_month = category_monthly_revenue.idxmax()
category_best_month_value = category_monthly_revenue.max()

category_worst_month = category_monthly_revenue.idxmin()
category_worst_month_value = category_monthly_revenue.min()

print(
    f"Electronics best day: {category_best_day} — "
    f"₹{category_best_day_value:,.2f} average revenue per transaction"
)
print(
    f"Electronics worst day: {category_worst_day} — "
    f"₹{category_worst_day_value:,.2f} average revenue per transaction"
)
print(
    f"Electronics best month: {category_best_month} — "
    f"₹{category_best_month_value:,.2f}"
)
print(
    f"Electronics worst month: {category_worst_month} — "
    f"₹{category_worst_month_value:,.2f}"
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    category_daily_revenue.index,
    category_daily_revenue.values,
    color="#2D1139"
)

plt.title('Electronics: Average Revenue per Transaction by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Revenue per Transaction (₹)')
plt.xticks(rotation=30)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    category_monthly_revenue.index.astype(str),
    category_monthly_revenue.values,
    color="#2D1139"
)

plt.title('Electronics: Total Revenue by Month')
plt.xlabel('Month')
plt.ylabel('Total Revenue (₹)')
plt.tight_layout()

plt.show()

### Electronics Monthly Revenue Trend

The original team analysis also included a separate line chart for Electronics monthly revenue from January to June 2026. This complements the monthly revenue table and bar chart by making the direction of the category trend easier to see.


In [ ]:
electronics_monthly_revenue = (
    df[df['category'] == 'Electronics']
    .groupby('month')['revenue']
    .sum()
)

plt.figure(figsize=(10, 5))
plt.plot(
    electronics_monthly_revenue.index.astype(str),
    electronics_monthly_revenue.values,
    marker='o'
)
plt.title('Electronics Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Electronics Trend Interpretation

The Electronics category shows a clear decline in monthly revenue from January to June 2026. This indicates that the category's performance weakened over the analysis period and should be investigated further before making major business decisions.


### Step 5.5 — Business Interpretation

For the overall chain, **Saturday is the strongest day**, with average revenue of about **₹4,420.02 per transaction**, while **Tuesday is the weakest**, at about **₹4,017.71**. The strongest month is **January (₹84.77 million)** and the weakest is **June (₹69.28 million)**.

Electronics follows a similar monthly pattern to the overall chain: **January is its strongest month at ₹45.69 million and June is its weakest at ₹29.69 million**. However, Electronics shows a different day-of-week pattern: **Wednesday is its strongest day (about ₹59,126.56 per transaction)** and **Tuesday is its weakest (about ₹54,306.75)**.

For promotional timing, the chain can consider stronger broad promotions around the weekend, especially Saturday, while Electronics-specific promotions may be better tested on Wednesday rather than assuming the overall chain pattern applies to every category. The persistent month-on-month decline in Electronics is more important than the day effect, so promotions should be targeted and measured for incremental revenue rather than simply increasing discount levels.

# Step 5.6 — Additional Total Revenue by Day-of-Week Analysis

The previous section measures **average revenue per transaction** by day of week.

This additional analysis measures **total revenue generated across all transactions** on each day of the week. Both metrics are useful, but they answer different questions:
- Average revenue per transaction shows the typical transaction value.
- Total revenue shows the overall revenue contribution of that weekday.


### Analyzing Revenue by Day of Week

I extracted the **day of the week** from the transaction date and created a `day_of_week` column. Then, I grouped the data by day and calculated the total revenue for each day. This helps me identify **weekly sales patterns** and understand which days perform better or weaker.


In [ ]:
df['day_of_week'] = df['date'].dt.day_name()

day_revenue = df.groupby('day_of_week')['revenue'].sum()

day_revenue

### Identifying the Best and Worst Performing Days

I identified the **highest-revenue** and **lowest-revenue** days using `idxmax()` and `idxmin()`. This helps me understand the weekly sales pattern and identify which days have the strongest and weakest overall revenue performance.


In [ ]:
best_day = day_revenue.idxmax()
best_day_revenue = day_revenue.max()

worst_day = day_revenue.idxmin()
worst_day_revenue = day_revenue.min()

print("Best Day:", best_day, "Revenue:", best_day_revenue)
print("Worst Day:", worst_day, "Revenue:", worst_day_revenue)

### Visualizing Revenue by Day of Week

I created a bar chart to visualize the **revenue distribution across the days of the week**. I arranged the days in their normal Monday-to-Sunday order so that the weekly pattern is easier to compare and interpret. This visualization helps highlight the strongest and weakest sales days clearly.


In [ ]:
import matplotlib.pyplot as plt

day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

day_revenue_ordered = day_revenue.reindex(day_order)

plt.figure(figsize=(10, 5))
plt.bar(day_revenue_ordered.index, day_revenue_ordered.values)

plt.title('Revenue by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Visualizing the Monthly Revenue Trend

I created a line chart to visualize how **total revenue changed month by month** from January to June 2026. The trend makes it easier to identify the overall direction of sales performance and quickly compare the highest and lowest revenue months.


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_revenue.index.astype(str),
    monthly_revenue.values,
    marker='o'
)

plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Summarizing the Time-Based Findings

I summarized the key **time-based sales findings** by displaying the best and worst performing months and days along with their revenue values. This gives a quick view of the **strongest and weakest revenue periods** and makes the main findings easier to use in the final business interpretation.


In [ ]:
print("===== TIME PATTERN SUMMARY =====")
print(f"Best Month  : {best_month} | Revenue: ₹{best_revenue:,.2f}")
print(f"Worst Month : {worst_month} | Revenue: ₹{worst_revenue:,.2f}")
print(f"Best Day    : {best_day} | Revenue: ₹{best_day_revenue:,.2f}")
print(f"Worst Day   : {worst_day} | Revenue: ₹{worst_day_revenue:,.2f}")

## Electronics — Total Revenue by Day of Week

The team also examined Electronics using **total revenue by weekday**. This complements the earlier Electronics analysis, which used average revenue per transaction.


### Electronics Day-wise Revenue Analysis

I focused on the **Electronics** category because it showed a significant decline in overall revenue. I filtered the Electronics transactions and calculated total revenue for each day of the week to identify its **strongest and weakest sales days**. This helps understand whether the category has any specific weekly sales pattern.


In [ ]:
electronics_day_revenue = (
    df[df['category'] == 'Electronics']
    .groupby('day_of_week')['revenue']
    .sum()
)

electronics_day_revenue

### Identifying Electronics' Best and Worst Days

I identified the **highest-revenue** and **lowest-revenue** days for Electronics using `idxmax()` and `idxmin()`. This helps me pinpoint the category's strongest and weakest weekly sales periods and use them in the final business recommendations.


In [ ]:
electronics_best_day = electronics_day_revenue.idxmax()
electronics_best_revenue = electronics_day_revenue.max()

electronics_worst_day = electronics_day_revenue.idxmin()
electronics_worst_revenue = electronics_day_revenue.min()

print("Electronics Best Day:", electronics_best_day, 
      "| Revenue:", electronics_best_revenue)

print("Electronics Worst Day:", electronics_worst_day, 
      "| Revenue:", electronics_worst_revenue)

### Visualizing Electronics Revenue by Day

I created a bar chart to visualize the **Electronics revenue pattern across the week**. I arranged the days from Monday to Sunday to make the comparison clear. This chart helps highlight the category's strongest and weakest sales days visually.


In [ ]:
electronics_day_revenue = electronics_day_revenue.reindex(day_order)

plt.figure(figsize=(10, 5))

plt.bar(
    electronics_day_revenue.index,
    electronics_day_revenue.values
)

plt.title('Electronics Revenue by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Final Team Observations

The merged analysis covers category trends, category consistency, product-level performance, discount patterns, monthly sales, and weekday sales behavior.

For weekday analysis, the notebook intentionally keeps **total revenue** and **average revenue per transaction** as separate measures. This avoids treating the two metrics as interchangeable.

The final project can therefore be presented as one connected EDA workflow rather than three separate notebooks.
